# Lab 1: Song Similarity with Audio-Feature Vectors

Author: Claude feat. Mike

Previously we used pretrained word embeddings (vectors) to explore dot products, cosine
similarity, and angles between vectors. 

In this lab you'll apply similar tools to a
different kind of vector: **songs**, represented by their Spotify audio features (danceability,
energy, loudness, tempo, etc.).

Dataset: [`spotify_songs.csv`](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-01-21/readme.md)
(TidyTuesday, Jan 2020), ~30,000 tracks pulled from Spotify playlists.

## Setup

Run the cells below as-is.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

### Load the data

Downloads the CSV the first time (and caches it in `../data/`, so later runs don't need the
network).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
DATA_PATH = Path("../data/spotify_songs.csv")

DATA_PATH.parent.mkdir(exist_ok=True)
if not DATA_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

songs = pd.read_csv(DATA_PATH)
print(f"loaded {len(songs)} rows")
songs.head()

- The first columns are more informational. 
- The columns at the end are the feature vectors we'll be using. 

### Cleaning the data

A couple of things need fixing before we can treat each row as one vector per song:

1. The same track shows up multiple times if it appears on multiple playlists (with identical
   audio features) — we only want one row per unique `track_id`.
2. A handful of rows are missing the track/artist name, which we need for displaying results.

In [ ]:
songs = songs.dropna(subset=["track_name", "track_artist"])
songs = songs.drop_duplicates(subset="track_id", keep="first")
songs = songs.reset_index(drop=True)

print(f"{len(songs)} unique songs remain")

### Build the feature matrix

We represent each song as a vector of its numeric audio features. Two things need handling:

- `key` (musical key, 0-11) and `mode` (major/minor, 0/1) are categorical labels, not
  continuous magnitudes, so we leave them out of the similarity vector.

In [ ]:
feature_columns = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_ms",
]

The remaining features are on very different scales — `duration_ms` is in the hundreds of
  thousands, `tempo` is in the tens/hundreds, while `danceability`, `energy`, etc. range from
  0 to 1:

In [ ]:
songs.hist(figsize=(16, 12), bins=30, layout=(4, 4));

A raw dot product would be dominated by whichever feature happens to have the largest
  numbers, which has nothing to do with musical similarity. We fix this by **standardizing**
  each feature (subtract its mean, divide by its standard deviation) so every feature
  contributes on a comparable scale.

In [ ]:

X = songs[feature_columns].to_numpy()
X = (X - X.mean(axis=0)) / X.std(axis=0)

print(f"feature matrix shape: {X.shape}")
# X[i] is now the vector for songs.loc[i]

In [ ]:
X.std(axis=0) # standard deviation is set to 1 for each feature

### Helper functions (provided)

A few small helpers for looking up songs and getting a link to listen on Spotify — each one is
followed by an example call showing what it does. (You'll write the dot-product /
cosine-similarity / angle functions yourself.)

In [ ]:
def find_song(name_contains, artist_contains=None):
    """Look up songs by a (partial, case-insensitive) name and optional artist."""
    matches = songs[songs["track_name"].str.contains(name_contains, case=False)]
    if artist_contains is not None:
        matches = matches[matches["track_artist"].str.contains(artist_contains, case=False)]
    return matches[["track_name", "track_artist", "track_id"]]

In [ ]:
matches = find_song("shape of you", "ed sheeran")
matches

In [ ]:
def spotify_url(track_id):
    return f"https://open.spotify.com/track/{track_id}"

In [ ]:
spotify_url(matches.iloc[0]["track_id"])

In [ ]:
def describe_song(index):
    row = songs.loc[index]
    return f"{row['track_name']} — {row['track_artist']} ({spotify_url(row['track_id'])})"

In [ ]:
describe_song(matches.index[0])

### Pick a target song

`find_song` can return more than one match (covers, remixes, re-releases all have their own
`track_id`) — inspect the results and pick the row you actually want.

In [ ]:
find_song("bad guy", "billie eilish")

In [ ]:
print(describe_song(43))

## Lab Exercises (the part where you do stuff)

### Question 0

Find a song that you want to work with, e.g. using `find_song`. Set `target_index` to its index so we can refer to it later.

In [ ]:
# TODO: set target_index

### Question 1

Find the song **most similar** to `target_index` (excluding the target song itself). Print its
name, artist, its Spotify link (`describe_song` gives you all three), and the angle (in degrees)
between it and the target song.

In [ ]:
# TODO: find the most similar song to `target_index` and print its name/artist/link and angle

### Question 2

Find the song **least similar** to `target_index` (again excluding the target song itself).
Print the same information: name, artist, Spotify link, and angle.

In [ ]:
# TODO: find the least similar song to `target_index` and print its name/artist/link and angle

### Question 3

Listen to (or think about) the target song, the most similar song, and the least similar song.
**Do the results make sense?** Justify your answer in terms of the actual audio features
involved — which features are close together / far apart between the songs?

_Your answer here._

### Question 4 (Extending)

The explanatory text above says we need "standardization". And yet, isn't the whole point of cosine similarity that the length of the vector doesn't matter, since we divide it out? Clarify the apparent contradiction.

_Your answer here._

### Question 5 (Extending)

We discarded two of the features, `key` and `mode` because they are categorical rather than numeric. Perhaps this information was crucial, though. How could we incorporate these features into our analysis? Please describe. If you'd like, try implementing it as well!